# 03_task3 — PA4 Task 3 (exact speculative sampling + LoRA policy distillation) on Colab L4

Target/teacher `Qwen/Qwen2.5-1.5B-Instruct`, draft/student `Qwen/Qwen2.5-0.5B-Instruct` (FP16, SDPA).
Run top-to-bottom (Colab "Runtime > Run all" on a fresh runtime; no edits, flags or restarts).
Every code cell prints `PASS` or `FAIL`. Setup / check cells (1-6) raise after `FAIL`, so Run all
stops there; experiment cells print `FAIL` and continue, so one failed stage does not block later,
independent stages.
Results go to Drive (`/content/drive/MyDrive/atml_pa4_results/task3`). Every stage is resumable:
finished units are skipped; a failed unit writes `error.json`, the driver continues, and the next run
of the cell retries it. A stored `check` / `validate` result that failed, and a `costs` summary that
lacks a draft that has since become available, are recomputed automatically.

| Cell | Stage | Est. L4 time |
|---|---|---|
| 1-4 | setup (incl. torchao removal) + CPU tests | ~6 min |
| 5 | `check`: tokenizer / chat-template agreement, split disjointness, bank prompts | 1 min |
| 6 | `validate`: chi^2 (gamma 2, 4) + negative control, split prefill, cache consistency | 3 min |
| 7 | `bank_spec`: exact_spec_original_gamma4 bank with our sampler | ~8 min |
| 8 | `pool`: frozen teacher rollouts for off-policy | ~5 min |
| 9 | `distill`: off/on-policy x seeds 6304, 6305 | ~40 min |
| 10 | `merge`: 4 merged FP16 checkpoints + merged-vs-unmerged check | ~3 min |
| 11 | `bank_students`: own-policy rollout bank per student | ~3 min |
| 12 | `banks_eval`: KL / TV / acceptance on every bank x 5 drafts | ~10 min |
| 13 | `costs`: isolated draft / target step costs (c, v_gamma) | ~3 min |
| 14 | `spec` original draft (step 2) | ~30 min |
| 15 | `spec` 4 merged drafts (step 6) | ~2 h |
| 16 | `unmerged`: one unmerged adapter, wrapper overhead | ~3 min |
| 17 | `standalone`: each draft's own accuracy (512 / 32 caps) | ~6 min |
| 18 | aggregate | ~2 min |

In [ ]:
# Cell 1: Drive + HF_HOME + data check
import os, sys, json
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

HF_HOME = '/content/drive/MyDrive/atml_pa4_hf_cache'
DATA_REPO = '/content/drive/MyDrive/atml-pa4-datasets'
DATA_ROOT = DATA_REPO + '/data/public_locked'
RESULTS_ROOT = '/content/drive/MyDrive/atml_pa4_results'
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(RESULTS_ROOT + '/task3', exist_ok=True)
os.environ['HF_HOME'] = HF_HOME

FILES = ['data/public_locked/gsm8k_train_locked.jsonl.gz', 'data/public_locked/gsm8k_test_locked.jsonl.gz', 'data/short_form/short_train.jsonl', 'data/short_form/short_test.jsonl', 'data/prefix_banks/prefix_banks_locked.jsonl.gz']
missing = [f for f in FILES if not os.path.exists(f'{DATA_REPO}/{f}')]
print('HF_HOME     :', HF_HOME)
print('DATA_REPO   :', DATA_REPO)
print('RESULTS_ROOT:', RESULTS_ROOT + '/task3')
ok = not missing
print('PASS' if ok else f'FAIL: missing {missing}')
if not ok:
    raise RuntimeError('Cell 1 FAIL: setup cell, stopping Run all here')

In [ ]:
# Cell 2: clone / pull repo
import subprocess
REPO_URL = 'https://github.com/reemwaheed13/ATML-PA4'
REPO_DIR = '/content/ATML-PA4'
if os.path.isdir(REPO_DIR + '/.git'):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], capture_output=True, text=True)
else:
    r = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
ok = r.returncode == 0 and os.path.exists(REPO_DIR + '/task3_specdec/run_task3.py')
if ok:
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    head = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
    print('HEAD:', head)
print('PASS' if ok else 'FAIL')
if not ok:
    raise RuntimeError('Cell 2 FAIL: setup cell, stopping Run all here')

In [ ]:
# Cell 3: pinned versions (handout: transformers==4.57.1, huggingface_hub==0.36.2) and no torchao.
# Colab ships torchao 0.10.0, which the installed peft rejects ("only versions above 0.16.0 are
# supported") when it creates any LoRA layer; nothing in this repo uses torchao, so it is removed.
# No earlier cell imports transformers / peft / torch in this kernel (every stage runs in a
# subprocess), so no runtime restart is needed.
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==4.57.1', 'huggingface_hub==0.36.2', 'peft>=0.14.0'], capture_output=True, text=True)
print(r.stderr[-2000:] if r.returncode else 'pip OK')
u = subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True, text=True)
print((u.stdout.strip() or u.stderr.strip())[-500:] if u.returncode == 0 else u.stderr[-2000:])
t = subprocess.run([sys.executable, '-c', "import importlib.util; print(importlib.util.find_spec('torchao') is None)"],
                   capture_output=True, text=True)
no_torchao = u.returncode == 0 and t.stdout.strip() == 'True'
print('torchao absent:', no_torchao)
v = subprocess.run([sys.executable, '-c',
                    'import transformers, huggingface_hub, torch;'
                    'print(transformers.__version__, huggingface_hub.__version__, torch.__version__, torch.cuda.is_available())'],
                   capture_output=True, text=True).stdout.split()
print('transformers, huggingface_hub, torch, cuda:', v)
ok = r.returncode == 0 and no_torchao and len(v) >= 4 and v[:2] == ['4.57.1', '0.36.2'] and v[3] == 'True'
print('PASS' if ok else 'FAIL')
if not ok:
    raise RuntimeError('Cell 3 FAIL: setup cell, stopping Run all here')

In [ ]:
# Cell 4: CPU unit tests
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], capture_output=True, text=True,
                   env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''})
print(r.stdout[-4000:])
print(r.stderr[-2000:])
ok = r.returncode == 0
print('PASS' if ok else 'FAIL')
if not ok:
    raise RuntimeError('Cell 4 FAIL: setup cell, stopping Run all here')

In [ ]:
# Cell 5: check: tokenizer / chat template agreement, train-test disjointness, bank prompt text
def run_driver(module, args):
    cmd = [sys.executable, '-u', '-m', module] + args
    print(' '.join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['check'] + ['--data-repo', DATA_REPO, '--results-root', T3])
ok = False
try:
    c = json.load(open(f'{T3}/check.json'))
    print('template mismatches:', c['chat_template_token_mismatches'], 'vocab equal:', c['vocab_equal'])
    print('splits:', c['splits']); print('banks:', c['supplied_banks'])
    print(c['example_prompt'])
    ok = rc == 0 and c['pass']
    print('PASS' if ok else 'FAIL')
except Exception as e:
    print(f'FAIL ({type(e).__name__}: {e})')
    raise
if not ok:
    raise RuntimeError('Cell 5 FAIL: setup/check cell, stopping Run all here')

In [ ]:
# Cell 6: validate: chi^2 on the shared round function + real-model cache checks
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['validate'] + ['--data-repo', DATA_REPO, '--results-root', T3])
ok = False
try:
    v = json.load(open(f'{T3}/sampler_validation.json'))
    for r in v['chi2']:
        print(f"gamma={r['gamma']} correct={r['correct_sampler']}: chi2={r['chi2']:.1f} dof={r['dof']} p={r['p_value']:.3g} "
              f"min expected={r['min_expected']:.0f} rejected={r['rejected_at_0.01']}")
    for c in v['real_model_checks']:
        print(c)
    ok = (rc == 0 and v['chi2_pass'] and all(c.get('argmax_equal', True) for c in v['real_model_checks'])
          and all(c.get('max_verify_logit_err_vs_full_forward', 0) < 0.5 for c in v['real_model_checks']))
    print('PASS' if ok else 'FAIL')
except Exception as e:
    print(f'FAIL ({type(e).__name__}: {e})')
    raise
if not ok:
    raise RuntimeError('Cell 6 FAIL: setup/check cell, stopping Run all here')

In [ ]:
# Cell 7: bank_spec: exact-speculative consistency bank (original draft, gamma 4, our sampler)
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['bank_spec'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    m = json.load(open(f'{T3}/banks/exact_spec_original_gamma4.meta.json'))
    print('records', m['n_records'], 'seconds', round(m['seconds']))
    print('PASS' if rc == 0 and m['n_records'] > 3000 else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 8: pool: frozen teacher rollouts (1 per math prompt, 8 per short prompt)
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['pool'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    m = json.load(open(f'{T3}/distill/teacher_pool.meta.json'))
    print(m)
    print('PASS' if rc == 0 and m['n_rollouts'] == 500 + 300 * 8 else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 9: distill: off/on-policy x seeds 6304, 6305 (40,000 teacher-scored tokens each)
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['distill'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    ok = rc == 0
    for n in ['off_s6304', 'off_s6305', 'on_s6304', 'on_s6305']:
        p = f'{T3}/distill/{n}/summary.json'
        if not os.path.exists(p):
            print(n, 'MISSING'); ok = False; continue
        s = json.load(open(p))
        print(n, 'updates', s['updates'], 'skipped', s['skipped_updates'], 'scored', s['scored_tokens'],
              'costs', {k: round(v or 0) for k, v in s['costs_s'].items()})
        ok &= s['scored_tokens'] == {'math': 30000, 'short': 10000} and s['updates'] == 157
    print('PASS' if ok else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 10: merge: LoRA -> fresh FP16 backbone (one checkpoint per run), merged vs unmerged logits
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['merge'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    ok = rc == 0
    for n in ['off_s6304', 'off_s6305', 'on_s6304', 'on_s6305']:
        p = f'{T3}/merge_logs/{n}/merge_summary.json'
        if not os.path.exists(p):
            print(n, 'MISSING'); ok = False; continue
        m = json.load(open(p)); print(n, m['verify'], m['seconds'])
        ok &= m['verify']['argmax_agreement'] > 0.99
    print('PASS' if ok else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 11: bank_students: each trained student's own frozen rollout bank
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['bank_students'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    import glob
    fs = sorted(glob.glob(f'{T3}/banks/student_*.jsonl.gz')); print(fs)
    print('PASS' if rc == 0 and len(fs) == 4 else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 12: banks_eval: alignment of every draft on every bank (same prefix IDs for all drafts)
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['banks_eval'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    s = json.load(open(f'{T3}/banks/alignment/summary.json'))
    print({k: s[k] for k in ('n_records', 'n_rows', 'drafts', 'banks')})
    print('PASS' if rc == 0 and s['n_rows'] == 5 * s['n_records'] and len(s['banks']) == 7 else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 13: costs: isolated decode-step / verification latency at cache lengths 64, 256, 1024
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['costs'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    s = json.load(open(f'{T3}/costs/summary.json'))
    for r in s['rows']:
        print(r)
    print('PASS' if rc == 0 and len(s['rows']) >= 6 else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 14: spec (step 2): original draft vs target-only, gamma 2 and 4, 120 prompts x 2 reps + fixed-64 diagnostic
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['spec', '--drafts', 'original'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    ok = rc == 0
    for g in (2, 4):
        p = f'{T3}/spec/original__g{g}/summary.json'
        ok &= os.path.exists(p)
        if os.path.exists(p):
            print(json.load(open(p)))
    print('PASS' if ok else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 15: spec (step 6): the four merged drafts, gamma 2 and 4 (resumable per draft x gamma)
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['spec', '--drafts', 'off_s6304,off_s6305,on_s6304,on_s6305'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    done = [n for n in ['off_s6304', 'off_s6305', 'on_s6304', 'on_s6305'] for g in (2, 4)
            if os.path.exists(f'{T3}/spec/{n}__g{g}/summary.json')]
    print(len(done), '/ 8 conditions')
    print('PASS' if rc == 0 and len(done) == 8 else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 16: unmerged: one unmerged LoRA adapter (off_s6304, gamma 4, 24 math prompts) - wrapper overhead
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['unmerged'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    print('PASS' if rc == 0 and os.path.exists(f'{T3}/spec/unmerged_off_s6304__g4/summary.json') else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 17: standalone: each draft as a model (512-token math / 32-token short caps, 2 reps)
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['standalone'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    import glob
    fs = sorted(glob.glob(f'{T3}/standalone/*/summary.json')); print(fs)
    print('PASS' if rc == 0 and len(fs) == 5 else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')

In [ ]:
# Cell 18: aggregate: every Task 3 table / plot
T3 = RESULTS_ROOT + '/task3'
rc = run_driver('task3_specdec.run_task3', ['aggregate'] + ['--data-repo', DATA_REPO, '--results-root', T3])
try:
    st = json.load(open(f'{T3}/tables/aggregate_status.json'))
    for k, v in st.items():
        print(k, v if not isinstance(v, dict) or 'traceback' not in v else v['error'])
    print('tables:', sorted(os.listdir(f'{T3}/tables')))
    print('PASS' if rc == 0 and all(v != 'missing' and not (isinstance(v, dict) and v.get('status') == 'failed')
                                    for v in st.values()) else 'FAIL')
except Exception as e:  # report FAIL; never stop Run all (later stages are independent)
    print(f'FAIL ({type(e).__name__}: {e})')